# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and analyze the FAIR2 colorectal cancer dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```
*Dataset: Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution*

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and prepare for usage with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print dataset summary
meta = dataset.metadata
print("Dataset loaded!")
print(f"Name: {meta.name}")
print(f"Identifier: {meta.identifier}")
print(f"Version: {meta.version}")
print(f"Description: {meta.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s using the Croissant metadata API.
The primary entry point for data is via *record sets*, which are structured tables or entities. Each record set has a unique `@id`.

In [ ]:
# List available record sets and their field @id's
print('Available record sets:')
recordset_ids = []
for recordset in dataset.metadata.record_sets:
    print(f"  - Name: {recordset.name}")
    print(f"    @id: {recordset.id}")
    recordset_ids.append(recordset.id)
    print(f"    Fields:")
    for field in recordset.fields:
        print(f"      - Name: {field.name}   |   @id: {field.id}    |   Data type: {field.data_type}")
    print()

## 3. Data Extraction
Extract data from the main record set(s) into a pandas DataFrame for analysis.

We use each table's unique `@id` to fetch records, as required by the Croissant specification.

Below we load all rows from all available record sets into DataFrames, using their `@id`s.

In [ ]:
# Extract all available record sets (@id's retrieved above)
dataframes = {}

for record_set_id in recordset_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}: {df.columns.tolist()}")
    print(f"Sample data (first 5 rows):")
    display(df.head())
    print("-----\n")

# For easy reference, pick first available record set as main for exploration.
main_record_set_id = recordset_ids[0]
main_df = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Apply common EDA operations:
- Filtering records based on a numeric field
- Normalizing numeric values
- Grouping/categorizing by a chosen field

We will use the record set and field `@id` values identified above. Adjust field `@id` and thresholds as appropriate for the dataset.

In [ ]:
# List columns and select candidate numeric and grouping fields (based on knowledge of medical datasets)
print("Columns in main record set:")
print(main_df.columns.tolist())

# Example: Assume '@id' of 'Age' is present and is numeric.
# You should select the corresponding exact '@id' from output above; here we search for it heuristically.
import re
num_field_candidates = [col for col in main_df.columns if re.search(r'age', col, re.IGNORECASE)]
if num_field_candidates:
    numeric_field_id = num_field_candidates[0]
else:
    # fallback: any float/int column
    numeric_field_id = main_df.select_dtypes(include=['float', 'int']).columns[0]

# Use Anatomical location, Sex, or MSI status as possible grouping fields, if available
group_field_candidates = [col for col in main_df.columns if re.search(r'anatomical|sex|msi|site|location', col, re.IGNORECASE)]
if group_field_candidates:
    group_field = group_field_candidates[0]
else:
    group_field = main_df.columns[0]

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field}")

# Filter records with Age > threshold, e.g., 60
threshold = 60
if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]):
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
else:
    print(f"Field {numeric_field_id} is not numeric! Skipping filter.")
    filtered_df = main_df.copy()

print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Add normalized numeric field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print(f"Cannot normalize non-numeric field {numeric_field_id}.")

# Group by selected field and show mean of numeric field
if group_field in filtered_df.columns and pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean {numeric_field_id} by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the numeric field (e.g., Age) and its relationship to the grouping variable (e.g., anatomical location/MSI status).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]):
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id], bins=12, kde=True, color='dodgerblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
else:
    print(f"Field {numeric_field_id} is not numeric; skipping distribution plot.")

# Boxplot of numeric field by group, if group field is categorical and not too many categories
if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) and main_df[group_field].nunique() < 10:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=main_df[group_field], y=main_df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.show()
else:
    print(f"Field {group_field} is not suitable for grouping or too many unique categories.")

## 6. Conclusion
- We have loaded a real-world, FAIR-compliant clinical oncology dataset defined via a Croissant schema and explored its tabular data using the `mlcroissant` Python library.
- Data are easily queried by record set and field `@id`.
- Basic filtering, normalization, grouping, and visualization allow for high-level insight before further modeling or statistical analysis.

For further analysis or publication, please ensure all interpretations appropriately consider the dataset's described limitations and scope.